<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/07_demo_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Démo live Gradio (Gemma E2B + LoRA)Interface de test en direct pour la soutenance : on tape une question de santé, on choisit la langue de réponse, et le modèle fine-tuné (Gemma E2B + LoRA) génère une réponse en direct. Case à cocher pour comparer avec la version avant fine-tuning (zero-shot), sans coût mémoire supplémentaire — même modèle, adaptateur simplement désactivé le temps de la comparaison.Scope volontairement limité à Gemma pour l'instant (pas TF-IDF/mT5/NLLB, qui demanderaient de centraliser les checkpoints des 3 sous-projets sur un même Drive avant de pouvoir les charger ici).Indépendant de `06_merge_adapter.ipynb` : ce notebook utilise l'adaptateur attaché (comme 04/05), pas le modèle fusionné — donc pas besoin d'attendre que la fusion soit terminée pour tester la démo.

In [ ]:
# Installe les dépendances (gradio pour l'interface, le reste comme 04/05).!pip install -q -U transformers accelerate peft bitsandbytes gradio

In [ ]:
# Monte Drive et localise l'adaptateur LoRA final produit en Phase 3.import osfrom google.colab import drivedrive.mount('/content/drive')  # demande l'autorisation d'accès au DrivePROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur DriveADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'  # adaptateur LoRA final de la Phase 3assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

In [ ]:
# Se connecte à Hugging Face avec le token des Colab Secrets.from google.colab import userdatafrom huggingface_hub import loginlogin(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [ ]:
# Charge le modèle de base en 4-bit et lui rattache l'adaptateur LoRA — identique à 04_evaluate.ipynb et# 05_generate_submission.ipynb, pour garantir exactement le même comportement que ce qui a été évalué.import gcimport torchfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfigfrom peft import PeftModelMODEL_NAME = "google/gemma-4-E2B-it"MAX_SEQ_LENGTH = 512  # même longueur maximale qu'à l'entraînementtokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer Gemmaif tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_token  # pas de token de padding défini : on réutilise le token de fingc.collect()  # libère la mémoire (Python, puis cache GPU)torch.cuda.empty_cache()bnb_config = BitsAndBytesConfig(  # même quantification 4-bit qu'à l'entraînement    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_compute_dtype=torch.bfloat16,    bnb_4bit_use_double_quant=True,)base_model = AutoModelForCausalLM.from_pretrained(  # charge le modèle de base quantifié    MODEL_NAME,    quantization_config=bnb_config,    torch_dtype=torch.bfloat16,    device_map={"": 0},  # tout sur le GPU 0, sans offload CPU)base_model.config.pad_token_id = tokenizer.pad_token_idbase_model.config.use_cache = True  # inférence : le cache KV accélère la générationdef unwrap_clippable_linears(model):    """Même déballage qu'en Phase 3/4/5 : nécessaire pour que PeftModel retrouve la structure    de modules sur laquelle l'adaptateur a été entraîné."""    count = 0    for module in model.modules():        for child_name, child in list(module.named_children()):            if child.__class__.__name__ == "Gemma4ClippableLinear":                setattr(module, child_name, child.linear)                count += 1    print(f'{count} couches Gemma4ClippableLinear déballées')    return modelbase_model = unwrap_clippable_linears(base_model)  # déballage avant de charger l'adaptateurmodel = PeftModel.from_pretrained(base_model, ADAPTER_DIR)  # rattache les poids LoRA au modèle de basemodel.eval()  # mode inférence (désactive le dropout)gc.collect()torch.cuda.empty_cache()print(f"Mémoire GPU allouée : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

In [ ]:
# Même template de prompt qu'en Phases 2/3/4/5, mais avec le nom de langue choisi directement dans# l'interface (pas besoin du code subset ici, puisqu'on ne travaille plus par sous-ensemble Val/Test).LANGUAGES = ['English', 'Amharic', 'Luganda', 'Swahili', 'Akan']  # dans l'ordre du menu déroulantdef build_prompt(question: str, language: str) -> str:    return (        f"Réponds à la question de santé suivante en {language}, "        f"de façon claire et médicalement fiable.\n\nQuestion : {question}"    )

In [ ]:
# Génération d'une seule réponse à la fois (pas de batch : ici on répond à une question tapée en direct).import time@torch.no_grad()def generate_one(chat_prompt, max_new_tokens=400):    inputs = tokenizer(chat_prompt, return_tensors='pt', add_special_tokens=False).to(model.device)    out = model.generate(        **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id,    )    new_tokens = out[:, inputs['input_ids'].shape[1]:]  # retire le prompt : ne garde que la réponse générée    return tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()def answer_question(question, language, compare_zero_shot):    prompt = build_prompt(question, language)    chat_prompt = tokenizer.apply_chat_template(        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True,    )    t0 = time.time()    finetuned = generate_one(chat_prompt)    t1 = time.time()    zero_shot, t_zs = None, None    if compare_zero_shot:        with model.disable_adapter():  # désactive temporairement LoRA : on retrouve le modèle de base            zero_shot = generate_one(chat_prompt)        t_zs = time.time() - t1    return finetuned, zero_shot, t1 - t0, t_zs

In [ ]:
# Interface Gradio : question + langue en entrée, réponse fine-tunée (+ zero-shot en option) en sortie.import gradio as grdef ui_fn(question, language, compare):    if not question or not question.strip():        return "Posez une question ci-dessus, puis cliquez sur Générer.", ""    finetuned, zero_shot, t_ft, t_zs = answer_question(question.strip(), language, compare)    out_ft = f"**Gemma fine-tuné** — généré en {t_ft:.1f} s\n\n{finetuned}"    out_zs = f"**Gemma zero-shot (avant fine-tuning)** — généré en {t_zs:.1f} s\n\n{zero_shot}" if compare else ""    return out_ft, out_zswith gr.Blocks(title="Démo Gemma — QA santé multilingue") as demo:    gr.Markdown(        "## Démo en direct — Gemma E2B + LoRA\n"        "Question de santé dans n'importe laquelle des 5 langues, réponse générée en direct par le modèle fine-tuné."    )    with gr.Row():        question_box = gr.Textbox(            label="Question de santé", placeholder="Ex. : Malaria ni nini?", lines=2, scale=3,        )        language_dd = gr.Dropdown(choices=LANGUAGES, value="English", label="Langue de réponse", scale=1)    compare_cb = gr.Checkbox(label="Comparer avec la version avant fine-tuning (zero-shot)", value=False)    submit_btn = gr.Button("Générer", variant="primary")    with gr.Row():        out_ft_md = gr.Markdown(label="Fine-tuné")        out_zs_md = gr.Markdown(label="Zero-shot")    submit_btn.click(ui_fn, inputs=[question_box, language_dd, compare_cb], outputs=[out_ft_md, out_zs_md])# share=True : lien public temporaire (~72h), utilisable par le jury depuis son propre appareil pendant# la soutenance. Lancer cette cellule 10-15 min avant de présenter (temps de chargement du modèle déjà fait# plus haut ; ce lancement lui-même est quasi instantané).demo.launch(share=True, debug=False)

---**Pour la soutenance:** lancer toutes les cellules à l'avance, garder cet onglet Colab ouvert pendant la présentation (le lien `share=True` meurt si la session s'arrête). Prévoir une courte vidéo de secours de la démo qui fonctionne, au cas où le Wi-Fi ou le quota GPU lâche au mauvais moment.**Amélioration optionnelle plus tard:** une fois `06_merge_adapter.ipynb` terminé, remplacer le bloc de chargement (base 4-bit + `PeftModel` + déballage) par un chargement direct depuis `checkpoints/gemma-4-e2b-merged` — plus rapide, sans dépendance à Hugging Face. La comparaison zero-shot perdrait alors son mécanisme actuel (`disable_adapter`) et demanderait de charger le modèle de base en plus, si vous voulez la garder.